<a href="https://colab.research.google.com/github/delaliajah/bestdeveloper/blob/master/Group_15304_GWP2_28_06_MScFE_620_Derivative_Pricing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MScFE 620 Deritives Pricing  - Group Work Project 1

Student Group:15304

Submission Date: 10th June 2026

$S_0=100; r=5\%; \sigma = 20\% ; T=3 \text{ months}$

5. We choose $N=100$ because the time step $\frac{T}{N}=\frac{3 \text{ months}=0.25 \text{ year}}{100}=0.0025 \text{ year} \approx 0.9 \text{ day}$  is good, because we normally assess financial data on daily basis.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import math

In [ ]:
# Create binomial tree function
def binomial_tree(S_ini, K, T, r, sigma, N, opttype):
    dt = T / N  # Define time step
    u = np.exp(sigma * np.sqrt(dt))  # Define u
    d = np.exp(-sigma * np.sqrt(dt))  # Define d
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    C = np.zeros([N + 1, N + 1])  # option prices
    S = np.zeros([N + 1, N + 1])  # underlying price

    for i in range(0, N + 1):
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
        if opttype == "C":
            C[N, i] = max(S[N, i] - K, 0)
        else:
            C[N, i] = max(K - S[N, i], 0)

    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i])  # Computing the European option prices
            S[j, i] = (S_ini * (u ** (i)) * (d ** (j - i)))  # Underlying evolution for each node
    return C[0, 0], C, S

In [ ]:
# Iterate N to find an optimal value, for ATM price
N_array = [1, 10, 20, 50, 100, 200, 400, 800]

for i in range(len(N_array)):
    call_price, C, S = binomial_tree(100, 100, 0.25, 0.05, 0.2, N_array[i], "C")
    put_price, P, S = binomial_tree(100, 100, 0.25, 0.05, 0.2, N_array[i], "P")
    print("With N = {:3d}, call price is {:.2f}, put price is {:.2f}".format(N_array[i], call_price, put_price))
    next

With N =   1, call price is 5.59, put price is 4.34
With N =  10, call price is 4.52, put price is 3.27
With N =  20, call price is 4.57, put price is 3.32
With N =  50, call price is 4.60, put price is 3.35
With N = 100, call price is 4.61, put price is 3.36
With N = 200, call price is 4.61, put price is 3.37
With N = 400, call price is 4.61, put price is 3.37
With N = 800, call price is 4.61, put price is 3.37


As above result, the call price and put price are stable from $N=100$, so 100 is confirmed to be a suitable number of step.

---



6. $\Delta$ is the rate of change of the options value following the change in stock price.

At $t=0$:

$\Delta _C = \frac{\delta C}{\delta S} = N(d_1)$

with
$d_1 = \frac{log(\frac{S_T}{K})+(\frac{r+\sigma ^2}{2})(T-t)}{\sigma \sqrt{T-t}}$

$\Delta _P = \Delta _C - 1$


In [ ]:
# Import Scipy library
import scipy.stats as ss

# Calculate d1 and delta
d1 = (np.log(100/100)+(0.05 + 0.2**2)/2*0.25)/(0.2*np.sqrt(0.25))
delta_call = ss.norm.cdf(d1)
delta_put = delta_call - 1
print("Delta for European call is {:.2f}".format(delta_call))
print("Delta for European put is {:.2f}".format(delta_put))

Delta for European call is 0.54
Delta for European put is -0.46


7. Vega is the sensitivity of options to changes in stock price volatility.

$\nu = \frac{\delta ^2V}{\delta \sigma} = SN'(d_1) \sqrt{T-t}$

In [ ]:
# Calculate Vega
vega = 100*ss.norm.pdf(d1)*np.sqrt(0.25)

print("Sensitivity of the option price to volatility is {:.2f}".format(vega))

Sensitivity of the option price to volatility is 19.82


In [ ]:
class TrinomialModel(object):  # Here we start defining our 'class' --> Trinomial Model!
    # First, a method to initialize our `TrinomialModel` algorithm!
    def __init__(self, S0, r, sigma, mat):
        self.__s0 = S0
        self.__r = r
        self.__sigma = sigma
        self.__T = mat

    # Second, we build a method (function) to compute the risk-neutral probabilities!
    def __compute_probs(self):
        self.__pu = ((np.exp(self.__r * self.__h / 2) - np.exp(-self.__sigma * np.sqrt(self.__h / 2)))
            / (np.exp(self.__sigma * np.sqrt(self.__h / 2)) - np.exp(-self.__sigma * np.sqrt(self.__h / 2)))) ** 2
        self.__pd = ((-np.exp(self.__r * self.__h / 2) + np.exp(self.__sigma * np.sqrt(self.__h / 2)))
            / (np.exp(self.__sigma * np.sqrt(self.__h / 2)) - np.exp(-self.__sigma * np.sqrt(self.__h / 2)))) ** 2
        self.__pm = 1 - self.__pu - self.__pd

        assert 0 <= self.__pu <= 1.0, "p_u should lie in [0, 1] given %s" % self.__pu
        assert 0 <= self.__pd <= 1.0, "p_d should lie in [0, 1] given %s" % self.__pd
        assert 0 <= self.__pm <= 1.0, "p_m should lie in [0, 1] given %s" % self.__pm

    # Third, this method checks whether the given parameters are alright and that we have a 'recombining tree'!
    def __check_up_value(self, up):
        if up is None:
            up = np.exp(self.__sigma * np.sqrt(2 * self.__h))

        assert up > 0.0, "up should be non negative"

        down = 1 / up

        assert down < up, "up <= 1. / up = down"

        self.__up = up
        self.__down = down

    # Four, we use this method to compute underlying stock price path
    def __gen_stock_vec(self, nb):
        vec_u = self.__up * np.ones(nb)
        np.cumprod(vec_u, out=vec_u)

        vec_d = self.__down * np.ones(nb)
        np.cumprod(vec_d, out=vec_d)

        res = np.concatenate((vec_d[::-1], [1.0], vec_u))
        res *= self.__s0

        return res

    # Fifth, we declare a Payoff method to be completed afterwards depending on the instrument we are pricing!
    def payoff(self, stock_vec):
        raise NotImplementedError()

    # Sixth, compute current prices!
    def compute_current_price(self, crt_vec_stock, nxt_vec_prices):
        expectation = np.zeros(crt_vec_stock.size)
        for i in range(expectation.size):
            tmp = nxt_vec_prices[i] * self.__pd
            tmp += nxt_vec_prices[i + 1] * self.__pm
            tmp += nxt_vec_prices[i + 2] * self.__pu

            expectation[i] = tmp

        return self.__discount * expectation

    # Seventh, Option pricing!
    def price(self, nb_steps, up=None):
        assert nb_steps > 0, "nb_steps shoud be > 0"

        nb_steps = int(nb_steps)

        self.__h = self.__T / nb_steps
        self.__check_up_value(up)
        self.__compute_probs()

        self.__discount = np.exp(-self.__r * self.__h)

        final_vec_stock = self.__gen_stock_vec(nb_steps)
        final_payoff = self.payoff(final_vec_stock)
        nxt_vec_prices = final_payoff

        for i in range(1, nb_steps + 1):
            vec_stock = self.__gen_stock_vec(nb_steps - i)
            nxt_vec_prices = self.compute_current_price(vec_stock, nxt_vec_prices)

            nxt_vec_prices = np.maximum(nxt_vec_prices,self.payoff(vec_stock))

        return nxt_vec_prices[0]

Step 2: Team B

Step 2: Question 16

In [ ]:
import numpy as np

def trinomial_tree_call(S0, K, r, sigma, T, N):
    dt = T / N
    # Standard Hull/Boyle Trinomial Parameters
    u = np.exp(sigma * np.sqrt(3 * dt))
    d = 1.0 / u
    m = 1.0

    # Probabilities
    pu = 1/6 + (r - 0.5 * sigma**2) * np.sqrt(dt / (3 * sigma**2))
    pd = 1/6 - (r - 0.5 * sigma**2) * np.sqrt(dt / (3 * sigma**2))
    pm = 2/3

    # Initialize stock prices at maturity
    num_nodes = 2 * N + 1
    S = np.zeros(num_nodes)
    for i in range(num_nodes):
        S[i] = S0 * (u ** (N - i))

    # Terminal payoffs for Call
    C = np.maximum(0, S - K)

    # Step backward
    discount = np.exp(-r * dt)
    for step in range(N - 1, -1, -1):
        num_nodes_step = 2 * step + 1
        C_next = np.zeros(num_nodes_step)
        for i in range(num_nodes_step):
            C_next[i] = discount * (pu * C[i] + pm * C[i+1] + pd * C[i+2])
        C = C_next
    return C[0]

# Standard baseline parameters
S0 = 100.0
r = 0.05
sigma = 0.20
T = 0.25
N = 3
strikes = [90, 95, 100, 105, 110]

# Generate data
prices = {K: trinomial_tree_call(S0, K, r, sigma, T, N) for K in strikes}

# Print Tabular Output
print(f"{'Strike (K)':<12} | {'Call Price':<12}")
print("-" * 27)
for K, price in prices.items():
    print(f"{K:<12} | {price:<12.2f}")


Strike (K)   | Call Price  
---------------------------
90           | 12.23       
95           | 8.40        
100          | 4.67        
105          | 2.93        
110          | 1.19        


Step 2: Question 16

In [ ]:
import numpy as np

def price_european_trinomial_put(S0, K, r, sigma, T, N):
    """
    Prices a European Put Option using a standard Cox-Ross-Rubinstein
    Trinomial Tree Framework.
    """
    dt = T / N

    # 1. Calculate step parameters (Up, Down, Flat scaling factors)
    u = np.exp(sigma * np.sqrt(3 * dt))
    d = 1.0 / u
    m = 1.0
    discount = np.exp(-r * dt)

    # 2. Calculate risk-neutral probabilities
    p_u = ((np.exp(r * dt / 2) - np.sqrt(d)) / (np.sqrt(u) - np.sqrt(d)))**2
    p_d = ((np.sqrt(u) - np.exp(r * dt / 2)) / (np.sqrt(u) - np.sqrt(d)))**2
    p_m = 1.0 - p_u - p_d

    # 3. Build the forward stock price grid
    S_grid = {}
    for i in range(N + 1):
        S_grid[i] = np.zeros(2 * i + 1)
        for j in range(2 * i + 1):
            # j=0 is the highest node (max up), moving down to 2i (max down)
            up_moves = i - j
            S_grid[i][j] = S0 * (u ** up_moves)

    # 4. Initialize option price dictionary and terminal payouts
    P_grid = {}
    P_grid[N] = np.maximum(0.0, K - S_grid[N])

    # 5. Step backward through time (Backward Induction)
    for i in range(N - 1, -1, -1):
        P_grid[i] = np.zeros(2 * i + 1)
        for j in range(2 * i + 1):
            # Node j at current step connects to j (Up), j+1 (Flat), and j+2 (Down) at step i+1
            hold = discount * (p_u * P_grid[i+1][j] + p_m * P_grid[i+1][j+1] + p_d * P_grid[i+1][j+2])
            P_grid[i][j] = hold

    # Return the value at root node (Time 0)
    return P_grid[0][0]

# --- Model Execution ---
# Define baseline market parameters
S0 = 100.0      # Starting stock price
r = 0.05        # Risk-free rate (5%)
sigma = 0.20    # Volatility (20%)
T = 0.25        # Expiration (3 months / 0.25 years)
N = 3           # Number of tree steps

# Define the 5 strike targets requested by prompt
strikes = {
    "Deep OTM (90%)": 90,
    "OTM (95%)": 95,
    "ATM (100%)": 100,
    "ITM (105%)": 105,
    "Deep ITM (110%)": 110
}

print(f"=== TRINOMIAL TREE EUROPEAN PUT RESULTS ===")
print(f"Baseline Stock Price S0: ${S0:.2f} | Time: {T} Year | Steps: {N}\n")

# Run loop to calculate and print results dynamically
for label, K in strikes.items():
    price = price_european_trinomial_put(S0, K, r, sigma, T, N)
    print(f"{label:<16} | Strike Price K: {K:>3} | Put Premium: ${price:.2f}")


=== TRINOMIAL TREE EUROPEAN PUT RESULTS ===
Baseline Stock Price S0: $100.00 | Time: 0.25 Year | Steps: 3

Deep OTM (90%)   | Strike Price K:  90 | Put Premium: $0.93
OTM (95%)        | Strike Price K:  95 | Put Premium: $2.44
ATM (100%)       | Strike Price K: 100 | Put Premium: $4.06
ITM (105%)       | Strike Price K: 105 | Put Premium: $7.22
Deep ITM (110%)  | Strike Price K: 110 | Put Premium: $10.38


17. The strike prices to be used are:

Deep OTM $K_1 = S_0 * 90\% = \$ 90$

OTM $K_2 = S_0 * 95\% = \$ 95$

ATM $K_3 = S_0 = \$ 100 $

ITM $K_4 = S_0 * 105\% = \$ 105$

Deep ITM $K_5 = S_0 * 110\% = \$ 110$




In [ ]:
class TrinomialAmericanCall(TrinomialModel):
    def __init__(self, S0, r, sigma, mat, K):
        super(TrinomialAmericanCall, self).__init__(S0, r, sigma, mat)
        self.__K = K

    def payoff(self, s):
        return np.maximum(s - self.__K, 0)

In [ ]:
K_array = [90, 95, 100, 105, 110]
for k in K_array:
  americantri = TrinomialAmericanCall(100, 0.05, .2, 0.25, k)
  print("For K = {:f}, the American Call price is {:.2f}".format(k,americantri.price(100)))

For K = 90.000000, the American Call price is 11.67
For K = 95.000000, the American Call price is 7.72
For K = 100.000000, the American Call price is 4.61
For K = 105.000000, the American Call price is 2.48
For K = 110.000000, the American Call price is 1.19


18

In [ ]:
class TrinomialAmericanPut(TrinomialModel):
    def __init__(self, S0, r, sigma, mat, K):
        super(TrinomialAmericanPut, self).__init__(S0, r, sigma, mat)
        self.__K = K

    def payoff(self, s):
        return np.maximum(self.__K - s, 0)

In [ ]:
for k in K_array:
  americantri = TrinomialAmericanPut(100, 0.05, .2, 0.25, k)
  print("For K = {:f}, the American Put price is {:.2f}".format(k,americantri.price(100)))

For K = 90.000000, the American Put price is 0.56
For K = 95.000000, the American Put price is 1.58
For K = 100.000000, the American Put price is 3.48
For K = 105.000000, the American Put price is 6.43
For K = 110.000000, the American Put price is 10.33


26

In [ ]:
# Create binomial tree function for american put
def american_put(S_ini, K, T, r, sigma, N):
    dt = T / N  # Define time step
    u = np.exp(sigma * np.sqrt(dt))  # Define u
    d = np.exp(-sigma * np.sqrt(dt))  # Define d
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    C = np.zeros([N + 1, N + 1])  # option prices
    S = np.zeros([N + 1, N + 1])  # underlying price
    delta = np.zeros([N,N])

    for i in range(0, N + 1):
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
        C[N, i] = max(K - S[N, i], 0)

    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i])  # Computing the European option prices
            S[j, i] = (S_ini * (u ** (i)) * (d ** (j - i)))  # Underlying evolution for each node
            C[j, i] = max(C[j, i], K - S[j, i])
            delta[j, i] = (C[j + 1, i + 1]-C[j + 1, i])/(S[j + 1, i + 1]-S[j + 1, i])
    return C[0, 0], C, S, delta

In [ ]:
put_price, P, S, delta = american_put(180, 182, 0.5, 0.02, 0.25, 25)
# Delta hedge dataframe
print(np.round(delta, 2))

[[-0.48  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.  ]
 [-0.56 -0.4   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.  ]
 [-0.65 -0.48 -0.32  0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.  ]
 [-0.73 -0.57 -0.39 -0.24  0.    0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.  ]
 [-0.81 -0.66 -0.48 -0.31 -0.18  0.    0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.  ]
 [-0.88 -0.75 -0.57 -0.39 -0.24 -0.12  0.    0.    0.    0.    0.    0.
   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
   0.  ]
 [-0.94 -0.83 -0.67 -0.49 -0.31 -0.17 -0.08  0.    0.    0.    0.    0.
   0.    0

In [ ]:
t_array = []
for i in range(26):
  t_array.append("t"+str(i))

In [ ]:
# Picking the path (d,d,d,...,d) (all down)
evolution = pd.DataFrame([S[:,0], P[:,0], np.concatenate((delta[:,0],[0]))], columns=t_array)
evolution.loc[3]=evolution.loc[0]*evolution.loc[2]
cash_acc = [-evolution.iloc[2][0]*evolution.iloc[0,0]+evolution.iloc[1,0]]
for i in range (1,26):
  cash_acc.append(-(evolution.iloc[2,i]-evolution.iloc[2,i-1])*evolution.iloc[0,i])

evolution.loc[4]=cash_acc
evolution.index = ['Stock price','Put option','Delta hedge','Stock portfolio value','Cash account']
# At t=25, the put option is exercised, we buy back 1 stock at strike price $182, and return it to the option dealer.
evolution.loc['Cash account','t25']=-182
total_cash = round(sum(evolution.loc['Cash account']),2)
evolution.loc[:, 'Total']=pd.Series(["","","","",total_cash], index=evolution.index)

/tmp/ipykernel_3110/4243063676.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cash_acc = [-evolution.iloc[2][0]*evolution.iloc[0,0]+evolution.iloc[1,0]]


In [ ]:
np.round(evolution,2)

,t0,t1,t2,t3,t4,t5,t6,t7,t8,t9,...,t17,t18,t19,t20,t21,t22,t23,t24,t25,Total
Stock price,180.00,173.75,167.71,161.89,156.26,150.83,145.59,140.54,135.65,130.94,...,98.68,95.26,91.95,88.75,85.67,82.69,79.82,77.05,74.37,
Put option,13.04,16.05,19.48,23.30,27.48,31.96,36.64,41.47,46.35,51.06,...,83.32,86.74,90.05,93.25,96.33,99.31,102.18,104.95,107.63,
Delta hedge,-0.48,-0.56,-0.65,-0.73,-0.81,-0.88,-0.94,-0.98,-1.00,-1.00,...,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00,0.00,
Stock portfolio value,-85.60,-97.44,-108.66,-118.67,-126.94,-133.07,-136.86,-138.34,-135.65,-130.94,...,-98.68,-95.26,-91.95,-88.75,-85.67,-82.69,-79.82,-77.05,0.00,
Cash account,98.64,14.81,14.60,13.79,12.39,10.54,8.41,6.23,2.12,-0.00,...,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-182.00,-0.46


27. We will use Hull-White method to calculate the average price used to determine the payoff for Asian Put option.

In [ ]:
S0 = 180
r = 0.02
sigma = 0.25
T = 0.5
K = 180
N = 25
M = 21  # number of representative average prices per node
dt = T / N  # Define time step
u = np.exp(sigma * np.sqrt(dt))  # Define u
d = np.exp(-sigma * np.sqrt(dt))  # Define d
p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs

# Precompute A_max and A_min at each node (step i, up moves J)
A_max=[[0]*(i + 1) for i in range(N + 1)]
A_min=[[0]*(i + 1) for i in range(N + 1)]

A_max[0][0]=S0
A_min[0][0]=S0
for i in range(1,N + 1):
    for J in range(i + 1):
        if J==0:
            term=S0*(d**i)
            A_max[i][J]= (i*A_max[i - 1][0] + term)/(i + 1)
            A_min[i][J]= (i*A_min[i - 1][0] + term)/(i + 1)
        elif J==i:
            term=S0*(u**i)
            A_max[i][J]= (i*A_max[i-1][J-1]+term)/(i + 1)
            A_min[i][J]= (i*A_min[i-1][J-1]+term)/(i + 1)
        else:
            term_up=S0*(u**(i-J)*d**J)
            term_down=S0*(u**(i-J)*d**J)
            A_max[i][J]= (i*A_max[i-1][J-1] + term_up)/(i + 1)
            A_min[i][J]= (i*A_min[i-1][J] + term_down)/(i + 1)

# tree_V[i][J][k] = option value at step i, J up moves, k-th representative average
tree_V = [[[0.0 for _ in range(M)] for _ in range(i + 1)] for i in range(N+1)]

# Terminal payoffs for Asian put: max(K - A, 0)
for J in range(N + 1):
    Amax = A_max[N][J]
    Amin = A_min[N][J]
    for k in range(M):
        A = (M - k)/M * Amax + k/M * Amin
        tree_V[N][J][k] = max(K - A, 0.0)

# Helper to find k such that A_bar lies between A_high and A_low
def find_k_bar(A_bar, step, J_bar):
    Amax_bar = A_max[step][J_bar]
    Amin_bar = A_min[step][J_bar]
    for km in range(1, M):
        A_high = (M - km)/M * Amax_bar + km/M * Amin_bar
        A_low = (M - (km - 1))/M * Amax_bar + (km - 1)/M * Amin_bar
        if A_low <= A_bar <= A_high:
            return km
    return M-1

def interp_weight(A_bar, step, J_bar, k_bar):
    Amax_bar = A_max[step][J_bar]
    Amin_bar=A_min[step][J_bar]
    A_high = (M - k_bar)/M * Amax_bar + k_bar/M * Amin_bar
    A_low = (M - (k_bar - 1))/M * Amax_bar + (k_bar - 1)/M * Amin_bar
    if A_high==A_low:
        return 0.5
    return (A_low - A_bar)/(A_low - A_high)

# Backward induction with Hull-White interpolation
for i in reversed(range(N)):
    for J in range(i + 1):
        for k in range(M):
            A = (M - k)/M * A_max[i][J] + k/M * A_min[i][J]
            # Up move average
            A_u = ((i + 1)*A + S0*(u**(i + 1-J)*d**J)) / (i + 2)
            # Down move average
            A_d = ((i + 1)*A + S0*(u**(i - J)*d**(J + 1))) / (i + 2)
            k_u = find_k_bar(A_u, i + 1, J)
            k_d = find_k_bar(A_d, i + 1, J + 1)
            w_u = interp_weight(A_u, i + 1, J, k_u)
            w_d = interp_weight(A_d, i + 1, J + 1, k_d)
            w_u = max(0.0, min(1.0, w_u))
            w_d = max(0.0, min(1.0, w_d))
            # Option values after up move
            if k_u==0:
                C_u = tree_V[i + 1][J][0]
            elif k_u>=M:
                C_u = tree_V[i + 1][J][M - 1]
            else:
                C_u = w_u * tree_V[i + 1][J][k_u] + (1-w_u) * tree_V[i + 1][J][k_u - 1]
            # Option values after down move
            if k_d==0:
                C_d = tree_V[i + 1][J + 1][0]
            elif k_d>=M:
                C_d = tree_V[i + 1][J + 1][M - 1]
            else:
                C_d = w_d * tree_V[i + 1][J + 1][k_d] + (1 - w_d) * tree_V[i + 1][J + 1][k_d - 1]
            tree_V[i][J][k] = math.exp(-r*dt)*(p*C_u + (1 - p)*C_d)

In [ ]:
# Calculate delta hedge
delta_tree = [[[0.0 for _ in range(M)] for _ in range(i + 1)] for i in range(N)]
for i in range(N):
    for J in range(i + 1):
        for k in range(M):
            A = (M - k)/M * A_max[i][J] + k/M * A_min[i][J]
            A_u = ((i + 1)*A + S0*(u**(i + 1 - J)*d**J)) / (i + 2)
            A_d = ((i + 1)*A + S0*(u**(i - J)*d**(J + 1))) / (i + 2)
            k_u = find_k_bar(A_u, i + 1, J)
            k_d = find_k_bar(A_d, i + 1, J + 1)
            w_u = interp_weight(A_u, i + 1, J, k_u)
            w_d = interp_weight(A_d, i + 1, J + 1, k_d)
            w_u = max(0.0, min(1.0, w_u))
            w_d = max(0.0, min(1.0, w_d))
            if k_u == 0:
                V_u = tree_V[i + 1][J][0]
            elif k_u >= M:
                V_u = tree_V[i + 1][J][M-1]
            else:
                V_u = w_u * tree_V[i + 1][J][k_u] + (1 - w_u) * tree_V[i + 1][J][k_u - 1]
            if k_d == 0:
                V_d = tree_V[i + 1][J + 1][0]
            elif k_d >= M:
                V_d = tree_V[i + 1][J + 1][M - 1]
            else:
                V_d = w_d * tree_V[i + 1][J + 1][k_d] + (1 - w_d) * tree_V[i + 1][J + 1][k_d - 1]
            S_u = S0 * (u**(i + 1 - J)) * d**J
            S_d = S0 * (u**(i - J)) * d**(J + 1)
            delta_u = 0.0 if S_u == S_d else (V_u - V_d) / (S_u - S_d)
            delta_tree[i][J][k] = delta_u
# Print delta hedge for all down
for i in range (25):
  print(delta_tree[i][0][0])

-0.017294287135677648
-0.012220831446699297
-0.008413511841696236
-0.005363507104849535
-0.003006801339739952
-0.001656201502582066
-0.000794352802394404
-0.00011208623339755468
0.00040810083213259964
0.0009791688893960547
0.001938749646089514
0.0037881292556182816
0.0073667475504944485
0.014318475266814642
0.02783029180260431
0.05409271080791639
0.1051380051385258
0.20428087397236444
0.3783015454199974
0.5779920803301782
0.6822459154429942
0.7082499423456191
0.6972230189227421
0.6234442420854871
0.4287464453946233


In [ ]:
# Picking the path (d,d,d,...,d) (all down)
evolution = pd.DataFrame([S[:,0], [tree_V[i][0][0] for i in range(26)], [delta_tree[j][0][0] for j in range(25)]], columns=t_array)
evolution.loc[3]=evolution.loc[0]*evolution.loc[2]
cash_acc = [-evolution.iloc[2][0]*evolution.iloc[0,0]+evolution.iloc[1,0]]
for i in range (1,26):
  cash_acc.append(-(evolution.iloc[2,i]-evolution.iloc[2,i-1])*evolution.iloc[0,i])

evolution.loc[4]=cash_acc
evolution.index = ['Stock price','Put option','Delta hedge','Stock portfolio value','Cash account']


/tmp/ipykernel_3110/4287850380.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cash_acc = [-evolution.iloc[2][0]*evolution.iloc[0,0]+evolution.iloc[1,0]]


In [ ]:
np.round(evolution,2)

,t0,t1,t2,t3,t4,t5,t6,t7,t8,t9,...,t16,t17,t18,t19,t20,t21,t22,t23,t24,t25
Stock price,180.00,173.75,167.71,161.89,156.26,150.83,145.59,140.54,135.65,130.94,...,102.23,98.68,95.26,91.95,88.75,85.67,82.69,79.82,77.05,74.37
Put option,0.34,0.22,0.14,0.09,0.05,0.03,0.01,0.01,0.01,0.01,...,1.17,2.36,4.74,9.33,16.58,25.45,34.99,44.73,53.75,60.19
Delta hedge,-0.02,-0.01,-0.01,-0.01,-0.00,-0.00,-0.00,-0.00,0.00,0.00,...,0.11,0.20,0.38,0.58,0.68,0.71,0.70,0.62,0.43,NaN
Stock portfolio value,-3.11,-2.12,-1.41,-0.87,-0.47,-0.25,-0.12,-0.02,0.06,0.13,...,10.75,20.16,36.04,53.14,60.55,60.68,57.66,49.76,33.03,NaN
Cash account,3.45,-0.88,-0.64,-0.49,-0.37,-0.20,-0.13,-0.10,-0.07,-0.07,...,-5.22,-9.78,-16.58,-18.36,-9.25,-2.23,0.91,5.89,15.00,NaN
